[⬅ Back to the examples](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/try/)

# Switches and selectors

Two-position switches keep their state; selectors choose one of several
positions; a spring-return position goes back to the rest position when
released. **Run all the cells**, then operate them with the mouse, by touch,
with the keyboard, or through the list under each selector.

**References:** IEC 60073 (lamp colors), ISA-5.1 (pump and motor symbols).
These standards inspired the widgets; the library does not claim conformity
with them ([Standards and
references](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/standards/)).

> **The code is hidden.** To see a cell's code, click the `⋯` bar above its result; to see all of it, choose **View ▸ Expand All Code**.

In [ ]:
# JupyterLite: install the package (bundled with this site) in the browser kernel
%pip install -q anywidget-instruments

In [ ]:
import anywidget_instruments as ai

ai.theme_switch()  # light / system / dark theme (STYLE-008)

## Two-position switches

In [ ]:
toggle = ai.ToggleSwitch(label="Pump P1")
rocker = ai.RockerSwitch(label="Heater")
slide = ai.SlideSwitch(True, label="Lighting")
key = ai.SelectorSwitch("OFF", positions=["OFF", "ON"], keyed=True, label="Maintenance key")
p1 = ai.LED(False, label="P1 running", size=(60, 60))
heating = ai.LED(False, on_color="#f59e0b", label="Heating", size=(60, 60))
lights = ai.LED(True, on_color="#fde047", label="Lights on", size=(60, 60))
maintenance = ai.LED(False, on_color="#3b82f6", label="Maintenance mode", size=(60, 60))
toggle.on_change(lambda change: setattr(p1, "value", change["new"]))
rocker.on_change(lambda change: setattr(heating, "value", change["new"]))
slide.on_change(lambda change: setattr(lights, "value", change["new"]))
key.on_change(lambda change: setattr(maintenance, "value", change["new"] == "ON"))
ai.Panel([toggle, rocker, slide, key, p1, heating, lights, maintenance], columns=4)

## Multi-position selectors

**Pump mode** is the usual HAND / OFF / AUTO selector. **Jog** has two
spring-return positions: REV and FWD hold only while you keep the selector
there, then it returns to 0.

In [ ]:
pump_mode = ai.SelectorSwitch("OFF", positions=["HAND", "OFF", "AUTO"], label="Pump mode")
jog = ai.SelectorSwitch(
    "0",
    positions=["REV", "0", "FWD"],
    spring_return=["REV", "FWD"],
    default_position="0",
    label="Jog",
)
pump = ai.Pump(mode="indicator", auto=False, label="Pump P2")
conveyor = ai.Motor(mode="indicator", label="Conveyor motor")


@pump_mode.on_change
def _(change):
    pump.value = "stopped" if change["new"] == "OFF" else "running"
    pump.auto = change["new"] == "AUTO"


jog.on_change(
    lambda change: setattr(
        conveyor, "value", {"REV": "reverse", "0": "stopped", "FWD": "forward"}[change["new"]]
    )
)
ai.Panel([pump_mode, jog, pump, conveyor], columns=4)

## Four-position rotary selector

A fan speed selector: OFF and three speeds. The gauge shows the fan speed.

In [ ]:
fan = ai.SelectorSwitch("OFF", positions=["OFF", "1", "2", "3"], label="Fan speed")
fan_rpm = ai.Gauge(0, max=2000, unit="rpm", format="%.0f", label="Fan speed (rpm)")
fan.on_change(
    lambda change: setattr(
        fan_rpm, "value", {"OFF": 0, "1": 600, "2": 1200, "3": 1800}[change["new"]]
    )
)
ai.Panel([fan, fan_rpm], columns=2)